# Notebook 1: Explainable AI (XAI) -- LIME on tabular data

## Course: Big Data Analytics
## University of Milano-Bicocca
## Degree Programme: Economics, Data Analysis and Management

---

### Introduction

A model can have a great accuracy and still be **right for the wrong reasons**. In this notebook we:

1. **train** a black-box model that decides on **mortgage applications** (Part 1);
2. **open** it with **LIME**, *Local Interpretable Model-agnostic Explanations*: first **by hand**, then with the library (Part 2);
3. **use** what LIME shows us to **audit and improve** the model (Part 3).

> 📚 **The series**: **1. LIME · tabular** (this notebook) → 2. LIME · text → 3. SHAP · tabular → 4. SHAP · text. Notebooks 1 and 3 explain **the same mortgage model**: same data, same training, only the explainer changes.

# IMAGINE: A BANK'S ALGORITHM REJECTS YOUR MORTGAGE. WOULD YOU ACCEPT "THE MODEL SAID NO" AS THE ONLY ANSWER?

---

## Why do we need explanations?

Random Forests, Gradient Boosting and neural networks are **black boxes**: we see the input and the output, not the reasoning. In credit decisions this is not enough:

- ⚖️ **Law**: in the EU, AI systems that assess creditworthiness are **high-risk** under the *AI Act*; in the US, a lender must tell a rejected applicant the **main reasons** for the denial.
- 🚫 **Fairness**: protected attributes (race, sex, ...) must not drive the decision, and we can only check this by looking inside the model.
- 🐛 **Debugging**: explanations reveal what the model has *really* learned.

> **Global** explanations describe the whole model; **local** explanations answer the question *"why THIS decision?"*. LIME is a **local** method.

In [ ]:
# ==========================================
# 1. SETUP: INSTALL AND IMPORT
# ==========================================

# LIME is not pre-installed on Colab: we install it with pip ("!" runs a shell command)
!pip install lime --quiet

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, roc_auc_score

from lime.lime_tabular import LimeTabularExplainer    # LIME for tables

warnings.filterwarnings("ignore")    # keep the output clean
np.random.seed(42)                   # reproducibility

# Colors of our own charts (colorblind-safe palette)
BLUE, ORANGE, RED, GRAY = "#2a78d6", "#eb6834", "#e34948", "#898781"
INK, GRID = "#52514e", "#e1e0d9"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_colwidth", 150)

def dumbbell(table, xlabel, title, fmt="{:.1%}"):
    # Helper for "before vs after" charts: one row per item, columns = [before, after]
    before, after = table.columns
    fig, ax = plt.subplots(figsize=(8, 1.2 + 0.55 * len(table)))
    for i, (a, b) in enumerate(table.values):
        ax.plot([a, b], [i, i], color="#c3c2b7", lw=2, zorder=1)
        ax.scatter(a, i, s=110, color=GRAY, edgecolors="white", lw=2, zorder=2, label=before if i == 0 else None)
        ax.scatter(b, i, s=110, color=BLUE, edgecolors="white", lw=2, zorder=3, label=after if i == 0 else None)
        ax.annotate(fmt.format(a), (a, i), xytext=(0, 8), textcoords="offset points", ha="center", color=INK, fontsize=8)
        ax.annotate(fmt.format(b), (b, i), xytext=(0, -15), textcoords="offset points", ha="center", color=INK, fontsize=8)
    ax.set_yticks(range(len(table)), table.index)
    ax.set_ylim(-0.7, len(table) - 0.3)
    ax.set_xlabel(xlabel)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt.format(v)))
    ax.set_title(title, loc="left", pad=26)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, ncol=2, loc="lower left", bbox_to_anchor=(0, 1.0))
    plt.tight_layout()
    plt.show()

def show_lime(explanation, **kwargs):
    # Same as explanation.show_in_notebook(), but works with every IPython version
    display(HTML(explanation.as_html(**kwargs)))

print("✅ Setup complete")

# 🏗️ Part 1 — Train the black box

This is the **training** of the mortgage model. **Notebook 3** (SHAP) repeats exactly this part (same data, same split, same random seeds), so it explains **the identical model**.

## 1.1 🏦 The data: mortgage applications

The **Boston HMDA** data (Munnell et al., 1996; also used in Stock & Watson's *Introduction to Econometrics*) contains **2,380 real mortgage applications** collected by the Federal Reserve Bank of Boston. We use 9 applicant characteristics:

| Feature | Meaning |
| :--- | :--- |
| `pirat` | debt payments / income (0.33 = 33% of the income goes to debt) |
| `hirat` | housing expenses / income |
| `lvrat` | loan / property value (*loan-to-value*) |
| `chist` | consumer credit history, from 1 (best) to 6 (worst) |
| `mhist` | mortgage credit history, from 1 (best) to 4 (worst) |
| `phist` | public bad credit record (e.g. bankruptcy)? 1 = yes |
| `selfemp` | self-employed? 1 = yes |
| `single` | single? 1 = yes |
| `afam` | African-American applicant? 1 = yes ⚠️ **protected attribute** |

**Target**: `deny` = 1 if the mortgage was denied.

In [ ]:
# ==========================================
# 2. LOAD THE MORTGAGE DATA
# ==========================================

URL_HMDA = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/HMDA.csv"
hmda = pd.read_csv(URL_HMDA)

# The 9 features we use. The file has a few more: e.g. 'insurance' (= a separate insurer
# refused the applicant) is almost a copy of the outcome, so we leave it out.
FEATURES = ['pirat', 'hirat', 'lvrat', 'chist', 'mhist', 'phist', 'selfemp', 'single', 'afam']

# yes/no columns -> 1/0
for col in ['deny', 'phist', 'selfemp', 'single', 'afam']:
    hmda[col] = (hmda[col] == 'yes').astype(int)

X = hmda[FEATURES]
y = hmda['deny']

print(f"{len(hmda)} applications | denied: {y.mean():.1%}")
X.head()

## 1.2 🌲 The black box: a Random Forest

A forest of 100 decision trees that vote together: accurate, but impossible to read.

In [ ]:
# ==========================================
# 3. TRAIN THE MORTGAGE BLACK BOX (RANDOM FOREST)
# ==========================================

# 75% train / 25% test, same denial rate in both sets (stratify), fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_test, y_test = X_test.reset_index(drop=True), y_test.reset_index(drop=True)  # applicants #0, #1, ...

def train_mortgage_model(X, y):
    # Our black box: a forest of 100 decision trees that vote together
    return RandomForestClassifier(n_estimators=100, min_samples_leaf=10, random_state=42).fit(X, y)

mortgage_model = train_mortgage_model(X_train, y_train)
p_deny = mortgage_model.predict_proba(X_test)[:, 1]      # P(deny) of every test applicant

auc = roc_auc_score(y_test, p_deny)
print(f"Accuracy: {accuracy_score(y_test, p_deny > 0.5):.3f}  (careful: approving everybody already gives {1 - y_test.mean():.3f})")
print(f"ROC-AUC:  {auc:.4f}  (0.5 = random ranking, 1 = perfect ranking)")
print(f"\n🔑 Fingerprint of the model: AUC = {auc:.4f} | P(deny) of applicant #112 = {p_deny[112]:.3f}")
print("   (identical in Notebooks 1 and 3: same data, same code, same seeds -> same model)")

The model ranks applicants well. **But why does it decide what it decides?** Let's open it.

# 🔍 Part 2 — How LIME works

## 2.1 The idea

> A complex model can be **locally** approximated by a simple one.

The Earth is round, but the street in front of your house looks flat. In the same way, the decision function of a black box is curved and irregular *globally*, but around **one** input it can be approximated by a **linear model**. The coefficients of that linear model are the explanation.

## 2.2 The formula

$$\xi(x) \;=\; \arg\min_{g \in G} \; \mathcal{L}\bigl(f, g, \pi_x\bigr) \;+\; \Omega(g)$$

- $f$: the black box; $x$: the instance to explain; $g \in G$: a simple (linear) model;
- $\mathcal{L}(f, g, \pi_x) = \sum_{z} \pi_x(z)\,\bigl(f(z) - g(z')\bigr)^2$: **local fidelity**, a weighted squared error over perturbed copies $z$ of $x$, where $z'$ is their 0/1 representation (for each feature: "is it still in the range of $x$?");
- $\pi_x(z) = \exp\!\bigl(-D(x,z)^2 / 2\sigma^2\bigr)$: **proximity kernel**, copies similar to $x$ weigh more ($D$ = distance, $\sigma$ = kernel width);
- $\Omega(g)$: **complexity** of $g$ (e.g. number of features used), kept small so that a human can read it.

## 2.3 The recipe in 4 steps (tables)

| Step | What LIME does to explain applicant $x$ |
| :--- | :--- |
| **1. Perturb** | create thousands of **copies** of $x$, replacing each feature with the value of a **random applicant** of the training data |
| **2. Predict** | ask the black box P(deny) for every copy |
| **3. Weight** | copies that kept **more features in the range of $x$** are closer to $x$ and weigh more |
| **4. Fit** | weighted linear regression: "feature still in the range of $x$" (1/0) → P(deny) |

**What is a "range"?** Each numeric feature is cut at its **quartiles** (4 ranges, each with 25% of the training applicants): e.g. `lvrat > 0.86` = the 25% of applicants with the highest loan-to-value. For a yes/no feature, the range is the value itself (e.g. `phist=yes`). LIME's question becomes: *"what happens to P(deny) when the loan-to-value is **not** in the top quartile any more?"*

The **coefficients** of step 4 are the explanation: **+** pushes toward denial, **−** toward approval.

## 2.4 🔧 LIME by hand (no library)

We explain applicant **#112** of the test set.

In [ ]:
# ==========================================
# 4. THE APPLICANT TO EXPLAIN
# ==========================================

APPLICANT = 112
applicant = X_test.iloc[APPLICANT]

display(X_test.iloc[[APPLICANT]])
print(f"True outcome: {'DENIED' if y_test[APPLICANT] else 'approved'} | Model: P(deny) = {p_deny[APPLICANT]:.3f}")

In [ ]:
# ==========================================
# 5. LIME BY HAND - STEP 0: THE RANGES OF THE APPLICANT
# ==========================================

NUMERIC = ['pirat', 'hirat', 'lvrat', 'chist', 'mhist']
BINARY  = ['phist', 'selfemp', 'single', 'afam']

# Numeric features are cut at their quartiles, computed on the training data
edges = {f: np.unique(np.percentile(X_train[f], [25, 50, 75])) for f in NUMERIC}

def range_index(f, values):
    # numeric feature -> number of its range (0 = lowest); yes/no feature -> the value itself
    return np.searchsorted(edges[f], values) if f in NUMERIC else np.asarray(values)

def range_name(f, value):
    # a readable name for the range of 'value', e.g. "lvrat > 0.86" or "phist=yes"
    if f in BINARY:
        return f"{f}={'yes' if value == 1 else 'no'}"
    e, k = edges[f], int(range_index(f, value))
    if k == 0:
        return f"{f} <= {e[0]:.2f}"
    if k == len(e):
        return f"{f} > {e[-1]:.2f}"
    return f"{e[k-1]:.2f} < {f} <= {e[k]:.2f}"

names = [range_name(f, applicant[f]) for f in FEATURES]
pd.DataFrame({"value": applicant.round(3), "range of the applicant": names})

In [ ]:
# ==========================================
# 6. LIME BY HAND - STEP 1 (PERTURB) AND STEP 2 (PREDICT)
# ==========================================

N_SAMPLES = 5000
rng = np.random.RandomState(0)

# STEP 1: every copy takes, feature by feature, the value of a RANDOM applicant of the training data
random_values = pd.DataFrame({f: rng.choice(X_train[f].values, N_SAMPLES - 1) for f in FEATURES})
copies = pd.concat([X_test.iloc[[APPLICANT]], random_values], ignore_index=True)   # row 0 = the applicant

# STEP 2: ask the black box
p_copies = mortgage_model.predict_proba(copies)[:, 1]

print(f"{N_SAMPLES:,} copies of applicant #{APPLICANT}. Row 0 is the applicant, rows 1-3 are three copies:")
copies.head(4).assign(**{"P(deny)": p_copies[:4].round(3)})

In [ ]:
# ==========================================
# 7. LIME BY HAND - STEP 3 (WEIGHT) AND STEP 4 (FIT)
# ==========================================

# STEP 3: Z[i, j] = 1 if feature j of copy i is still in the applicant's range, 0 if it moved
Z = pd.DataFrame({f: (range_index(f, copies[f]) == range_index(f, applicant[f])).astype(int) for f in FEATURES})
distance = np.sqrt((1 - Z).sum(axis=1).values)           # sqrt(number of features that moved)
KERNEL_WIDTH = 0.75 * np.sqrt(len(FEATURES))              # LIME's default width
weights = np.exp(-distance**2 / (2 * KERNEL_WIDTH**2))    # Gaussian kernel pi_x(z)

# STEP 4: weighted linear regression   "feature in the applicant's range (1/0)"  ->  P(deny)
surrogate = Ridge(alpha=1.0).fit(Z, p_copies, sample_weight=weights)

lime_by_hand = pd.Series(surrogate.coef_, index=names).sort_values(key=abs, ascending=False)
print("LIME weights (by hand):   + pushes toward DENIAL,  - pushes toward APPROVAL\n")
print(lime_by_hand.round(3).to_string())

In [ ]:
# ==========================================
# 8. WHAT LIME SEES
# ==========================================

# For every feature: weighted average P(deny) of the copies that LEFT the applicant's range
# vs the copies that KEPT it
avg = lambda mask: np.average(p_copies[mask], weights=weights[mask])
sees = pd.DataFrame({
    "feature moved to another range": [avg((Z[f] == 0).values) for f in FEATURES],
    "feature in the applicant's range": [avg((Z[f] == 1).values) for f in FEATURES]}, index=names)
gap = sees.iloc[:, 1] - sees.iloc[:, 0]

dumbbell(sees.loc[gap.sort_values().index], "weighted average P(deny) of the copies",
         f"What LIME sees: {N_SAMPLES:,} copies of applicant #{APPLICANT}")

print("The gap between the two dots is (almost) the LIME weight:\n")
print(pd.DataFrame({"gap": gap, "LIME weight": lime_by_hand}).loc[lime_by_hand.index].round(3).to_string())

### 💡 What did we just see?

For each feature, LIME compares the copies that **keep** the applicant's range (blue) with the copies that **leave** it (gray). For example, copies with a public bad credit record (`phist=yes`) have an average P(deny) of about 25%, the others about 11%: the gap, **+0.14**, is the LIME weight of `phist=yes`.

**That is all LIME does**: perturb → predict → weight → fit.

## 2.5 📦 The same with the `lime` library

`LimeTabularExplainer` runs the same 4 steps. One detail differs: inside each range, the library draws the value from a **bell curve** fitted on the training applicants of that range, while we reused **real values**.

In [ ]:
# ==========================================
# 9. LIME WITH THE LIBRARY
# ==========================================

binary_idx = [FEATURES.index(f) for f in ['phist', 'selfemp', 'single', 'afam']]

def predict_mortgage(data):
    # LIME passes a numpy array: we give the model its column names back
    return mortgage_model.predict_proba(pd.DataFrame(data, columns=FEATURES))

tab_explainer = LimeTabularExplainer(
    X_train.values, feature_names=FEATURES, class_names=['approved', 'denied'],
    categorical_features=binary_idx, categorical_names={i: ['no', 'yes'] for i in binary_idx},
    random_state=42)

exp_app = tab_explainer.explain_instance(applicant.values, predict_mortgage, num_features=9, num_samples=5000)

# Interactive view: predicted probabilities | feature weights | values of the applicant
show_lime(exp_app, show_table=True)

In [ ]:
# ==========================================
# 10. BY HAND vs LIBRARY
# ==========================================

library = pd.Series(dict(exp_app.as_list()))
print(pd.DataFrame({"by hand": lime_by_hand, "lime library": library}).loc[library.index].round(3), "\n")

exp_app.as_pyplot_figure().set_size_inches(7, 4)
plt.show()
print(f"Black box P(deny): {p_deny[APPLICANT]:.3f} | LIME's local linear model: {exp_app.local_pred[0]:.3f} "
      f"| local R²: {exp_app.score:.2f}")

### 💡 How to read it

- **Same story, by hand and with the library**: a public bad credit record (`phist=yes`), a high loan-to-value (`lvrat > 0.86`) and a bad credit history (`chist > 2.00`) push toward **denial** (green bars); a moderate debt ratio (`0.28 < pirat <= 0.33`) pushes toward approval (red bars).
- The **numbers** are not identical because the two versions build the neighbourhood differently (real values vs bell curve): LIME's answer **depends on how the copies are generated**.
- **`afam=yes` also pushes toward denial**: the applicant's **race** is among the reasons.
- ⚠️ LIME's linear model only **approximates** the black box (compare the two predictions above and the low local $R^2$). LIME weights give the *direction* and a *rough size* of each effect; they do **not** add up to the prediction (SHAP does: Notebook 3).

# QUESTION: `afam` IS THE APPLICANT'S RACE. IS IT ACCEPTABLE THAT IT PUSHES A MORTGAGE TOWARD DENIAL, EVEN A LITTLE?

### 💡 From the mechanism to the practice

| Because LIME... | ...it can | ...but |
| :--- | :--- | :--- |
| only needs `predict_proba` on perturbed copies | explain **any** model (forest, boosting, network, a model behind an API) | every explanation costs thousands of model calls |
| asks "what if this feature left the applicant's range?" | give **reasons** in the applicant's terms (`lvrat > 0.86`) | the answer depends on how ranges and copies are built |
| resamples every feature **independently** | — | creates **unrealistic** applicants (Part 4) |
| fits a **linear** model on a **neighbourhood** | give a short, readable list of +/− weights | it is only an approximation (local $R^2$) and the weights do not add up to the prediction |
| uses **random** sampling | — | results change with `num_samples` and the seed (Part 4) |

# ⚖️ Part 3 — Auditing the model with LIME

## 3.1 Is applicant #112 a one-off?

One explanation is an anecdote. Let's explain **all** the African-American applicants of the test set (a few seconds).

In [ ]:
# ==========================================
# 11. LIME FOR ALL AFRICAN-AMERICAN APPLICANTS
# ==========================================

afam_applicants = np.where(X_test['afam'] == 1)[0]
afam_weight, afam_rank = [], []
for i in afam_applicants:
    e = tab_explainer.explain_instance(X_test.iloc[i].values, predict_mortgage, num_features=9, num_samples=2000)
    reasons = [name for name, w in e.as_list()]               # sorted from the most important
    afam_rank.append(reasons.index("afam=yes") + 1)
    afam_weight.append(dict(e.as_list())["afam=yes"])
afam_weight = np.array(afam_weight)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(afam_weight, bins=20, color=ORANGE)
ax.axvline(0, color=INK, lw=1)
ax.set_xlabel("LIME weight of 'afam=yes'   (+ = toward denial)")
ax.set_ylabel("applicants")
ax.set_title(f"All {len(afam_applicants)} African-American applicants of the test set", loc="left")
plt.show()

print(f"'afam=yes' pushes toward denial for {np.mean(afam_weight > 0):.0%} of them "
      f"(average weight {afam_weight.mean():+.3f}); median rank among the 9 reasons: {np.median(afam_rank):.0f}")

💡 Race is rarely the **main** reason, but it pushes toward denial in practically **every** explanation: the problem is **systematic**, not a one-off.

## 3.2 Does race change decisions?

In [ ]:
# ==========================================
# 12. WHAT IF ONLY THE RACE WERE DIFFERENT?
# ==========================================

twin = X_test.iloc[[APPLICANT]].copy()
twin['afam'] = 0                                  # identical applicant, different race
p_twin = mortgage_model.predict_proba(twin)[0, 1]

decision = lambda p: "DENIED" if p > 0.5 else "approved"
print(f"Applicant #{APPLICANT} (afam = 1): P(deny) = {p_deny[APPLICANT]:.3f} -> {decision(p_deny[APPLICANT])}")
print(f"Identical twin (afam = 0): P(deny) = {p_twin:.3f} -> {decision(p_twin)}")

🚨 Same income, same debts, same credit history: **only the race changes, and so does the decision**.

The model learned this from the historical decisions contained in the data. Using race in credit decisions is illegal (in the EU and in the US), and without an explanation nobody would have noticed.

## ✍️ Exercise — A fairer model?

1. Put the protected attribute in `features_to_drop`.
2. Re-run the two cells: the model is retrained without it, and we compare the **AUC**, the **decision for applicant #112** and the **average predicted risk** of the two groups.

In [ ]:
# ==========================================
# 13. ✍️ EXERCISE: YOUR FIX
# ==========================================

features_to_drop = []        # 👉 e.g. ['feature_name']

kept = [f for f in FEATURES if f not in features_to_drop]
new_mortgage_model = train_mortgage_model(X_train[kept], y_train)
p_new = new_mortgage_model.predict_proba(X_test[kept])[:, 1]

In [ ]:
# ==========================================
# 14. ORIGINAL vs NEW MODEL
# ==========================================

print(f"Dropped features: {features_to_drop}")
print(f"ROC-AUC:             original {roc_auc_score(y_test, p_deny):.3f} | new {roc_auc_score(y_test, p_new):.3f}")
print(f"Applicant #{APPLICANT}:      original {p_deny[APPLICANT]:.3f} ({decision(p_deny[APPLICANT])}) | "
      f"new {p_new[APPLICANT]:.3f} ({decision(p_new[APPLICANT])})\n")

group = X_test['afam'].map({0: "afam = 0", 1: "afam = 1 (African-American)"})
risk = pd.DataFrame({"Original model": pd.Series(p_deny).groupby(group).mean(),
                     "New model": pd.Series(p_new).groupby(group).mean()})
dumbbell(risk, "average predicted P(deny)", "Average predicted risk of denial, by group")

if not features_to_drop:
    print("ℹ️ features_to_drop is empty: the new model is identical to the original. Drop the protected attribute!")

<details>
<summary>✅ <b>Solution</b> (click to open)</summary>

```python
features_to_drop = ['afam']
```

- The AUC drops by about 1 point: in the historical data race is correlated with the outcome, so it carried some (unacceptable) predictive information.
- Applicant #112 is now **approved**.
- The gap between the two groups **shrinks but does not disappear**: the other features (credit history, loan-to-value, ...) also differ between the groups in the data.

</details>

# QUESTION: THE GAP DID NOT DISAPPEAR. ARE THE REMAINING DIFFERENCES LEGITIMATE RISK FACTORS OR *PROXIES* OF RACE? HOW COULD YOU FIND OUT?

# ⚠️ Part 4 — The limits of LIME

## 4.1 Is LIME stable?

LIME learns from **random** copies: with too few of them, the explanation changes from run to run. Let's explain applicant #112 three times (3 seeds) with 100 and with 5,000 copies.

In [ ]:
# ==========================================
# 15. IS LIME STABLE?
# ==========================================

rows = []
for n in [100, 5000]:
    for seed in [0, 1, 2]:
        explainer = LimeTabularExplainer(
            X_train.values, feature_names=FEATURES, class_names=['approved', 'denied'],
            categorical_features=binary_idx, categorical_names={i: ['no', 'yes'] for i in binary_idx},
            random_state=seed)
        e = explainer.explain_instance(applicant.values, predict_mortgage, num_features=9, num_samples=n)
        rows.append({"num_samples": n, "seed": seed,
                     "top-3 reasons": ",  ".join(f"{name} ({w:+.2f})" for name, w in e.as_list()[:3])})
pd.DataFrame(rows)

With 100 copies the top reasons change at every run; with 5,000 they are stable (two reasons with almost the same weight can still swap places).

## 4.2 Are the copies realistic?

`pirat` includes **all** the debt payments, housing included, so housing expenses (`hirat`) cannot be larger than `pirat`.

In [ ]:
# ==========================================
# 16. ARE LIME'S COPIES REALISTIC?
# ==========================================

real = (X_train['hirat'] > X_train['pirat']).mean()
fake = (copies['hirat'] > copies['pirat']).mean()

print("Applicants whose housing expenses are larger than ALL their debt payments (hirat > pirat):")
print(f"   real applicants:            {real:.1%}")
print(f"   LIME's copies (Part 2.4):   {fake:.1%}")

LIME resamples every feature **independently**, so about one copy in five is an applicant that cannot exist, and the black box is asked to predict on it.

**The limits of LIME** follow from its mechanism:
- **Instability**: use enough `num_samples` and check with several seeds.
- **Unrealistic copies**: independent resampling ignores the links between features.
- **The neighbourhood is a choice**: ranges, sampling and kernel width change the numbers (by hand vs library, Part 2).
- **Approximation**: the linear model may fit the black box poorly (local $R^2$); its weights do not add up to the prediction.
- **Locality**: one explanation = one applicant; no global view of the model.

# 📌 Summary

| What LIME revealed | What we did | Result |
| :--- | :--- | :--- |
| The **reasons** for the denial of #112: public record, loan-to-value, credit history | — | an explanation the bank can give to the applicant |
| **Race** pushes toward denial for practically every African-American applicant, and flips the decision on #112 | Drop the protected attribute | AUC −1 point, #112 approved, group gap reduced (not zero) |

### 🔑 The golden rule

> Accuracy tells you **how often** a model is right. Explanations tell you **why**, and whether that *why* is acceptable.

➡️ **Notebook 2** applies LIME to **text**; **Notebook 3** explains **this same model** with **SHAP**, whose contributions **add up exactly** to the prediction.

## 🎯 Final exercises

1. **What-if for applicant #112.** Lower `lvrat` from 0.90 to 0.75 and predict again. Is the change consistent with LIME's weight for `lvrat > 0.86`?
2. **Kernel width.** In the LIME-by-hand cells, try `KERNEL_WIDTH = 0.5` and `20`. How do the weights change, and why? *(Hint: what does a very wide kernel do to the "local" in LIME?)*
3. **An approved applicant.** Explain an African-American applicant who was **approved** (e.g. #10). Which features push toward approval? Does `afam=yes` still push toward denial?
4. **Another neighbourhood.** In Step 1, draw the random values only from the **denied** applicants (`X_train[y_train == 1]`). How do the weights change? What does this tell you about LIME?

---

### References

- Ribeiro, M. T., Singh, S., Guestrin, C. (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*. KDD 2016.
- Munnell, A. H., Tootell, G. M. B., Browne, L. E., McEneaney, J. (1996). *Mortgage Lending in Boston: Interpreting HMDA Data*. American Economic Review, 86(1), 25-53.
- Stock, J. H., Watson, M. W. *Introduction to Econometrics* (chapter on binary dependent variables, Boston HMDA data).
- LIME library: <https://github.com/marcotcr/lime>